# 为营销活动和演示文稿生成透明图像素材

一张好图应该能适配任何背景。透明图像素材让你可以在营销活动、网站、演示文稿和产品目录中轻松复用同一套视觉元素，而不会出现白色方框，也无需手动去除背景。

本 cookbook 涵盖四个客户用例：

- **季节性电商营销活动：** 一次生成透明的产品图片，然后在店铺背景、季节性横幅和营销版面中重复使用。
- **品牌化的企业演示文稿：** 生成透明图表，使其直接融入规定的 PowerPoint 主题和渐变幻灯片背景。
- **设计模板素材：** 生成应用图标、贴纸和装饰元素，供用户放置在任意模板或背景上。
- **按需印刷商品：** 将同一个透明印花设计应用到不同的空白服装和产品样机上。

## 设置

使用 `pip install openai pillow` 安装 Python 包，并在环境中设置 `OPENAI_API_KEY`。你还需要能够访问支持透明背景的图像模型，以及用于构建网站的 Codex。

本示例使用 `gpt-image-2.5-flare`，并通过 `background="transparent"` 请求 PNG 输出。如果想与针对高质量要求优化的模型进行比较，请将 `IMAGE_MODEL` 设置为 `"gpt-image-2.5-sunburst"`。两个模型都支持透明背景。第一次比较时保持提示、尺寸和 `quality="high"` 不变，先检查透明度、细微边缘和文字，再调整质量或延迟。

下面的图片和截图保留自本 cookbook 最初的 GPT Image 2 版本。它们用于说明工作流，并非 GPT Image 2.5 的实测结果。在重新运行生成单元格之前，请先保存这些素材的副本，因为生成单元格会写入相同的文件名。

更多背景信息，请参阅[图像提示指南](https://developers.openai.com/api/docs/guides/image-prompting)、[OpenAI 模型页面](https://developers.openai.com/api/docs/models)和[图像生成指南](https://developers.openai.com/api/docs/guides/image-generation)。

In [ ]:
import base64
import os
from pathlib import Path

from openai import OpenAI
from PIL import Image

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

IMAGE_MODEL = "gpt-image-2.5-flare"

client = OpenAI()
asset_dir = Path("images/transparent-image-assets")
asset_dir.mkdir(parents=True, exist_ok=True)

## 用例 1：季节性电商营销活动

一个虚构的家居香氛品牌希望在多个季节性店铺页面中复用同一个产品系列。一次生成透明的产品图，然后把同样的素材放到不同的营销背景上，无需手动抠图。

### 1.1 定义原始产品提示

网站使用同一个虚构家居香氛品牌的四款产品。每条原始产品描述后面都跟着相同的品牌和透明度指令，以保持整个系列的视觉一致性。

**注意：** 提示指令的优先级高于 `background="transparent"`。如果提示中描述了背景幕布、场景、颜色或其他背景，模型可能会生成该背景，而不是生成透明效果。请让提示聚焦于孤立的主体，并明确要求透明背景。

In [ ]:
brand_prompt = (
    "One object from STILLROOM, an understated luxury botanical home-fragrance "
    "atelier: artisanal warm amber and honey-tinted hand-blown glass, subtle "
    "brushed brass hardware, natural ivory cotton cord, quiet "
    "Japanese-Scandinavian restraint, warm side-lit editorial studio "
    "photography, photorealistic micro-detail, immaculate premium campaign "
    "quality. Full object completely visible and generously padded. Preserve "
    "every natural transparency, refraction, translucent layer and fine "
    "material edge. Output an isolated object on actual fully transparent "
    "alpha; no backdrop, no rectangle, no plinth, no cast shadow, no readable "
    "writing, no label text, no watermark."
)

products = {
    "amber-parfum": (
        "A sculptural substantial clear honey-amber faceted glass "
        "eau-de-parfum bottle, slender rectangular body with softly rounded "
        "shoulders, heavy clear crystal cap, a small delicate sheer ivory "
        "organza ribbon loosely tied around the brushed brass neck with one "
        "translucent airy tail floating naturally; visible golden fragrance "
        "liquid and beautifully realistic glass refraction."
    ),
    "botanical-oil": (
        "A slim elegant cylindrical deep honey-amber transparent hand-blown "
        "glass botanical body-oil bottle, tall slender polished brushed-brass "
        "dropper cap with tiny ivory rubber bulb, translucent pale golden "
        "botanical oil inside and a tiny delicate preserved sprig of green "
        "wild fennel visible suspended within the liquid; subtle refraction, "
        "premium apothecary design."
    ),
    "reed-diffuser": (
        "A rounded low smoked-amber translucent glass home-fragrance diffuser "
        "vessel containing warm golden scented oil, topped by five naturally "
        "splayed very thin blonde rattan reeds and one extraordinarily "
        "delicate feathery dried pampas grass stem with fine hairlike wisps "
        "extending outward; realistic transparent glass and botanical fibers."
    ),
    "garden-candle": (
        "A quiet luxurious handmade botanical candle in a thick heavy "
        "translucent smoky-amber and milky frosted glass tumbler, creamy "
        "natural ivory wax, one small gently glowing clean candle flame, "
        "delicate transparent-edged frosted glass rim, a loose fine ivory "
        "cotton cord tied around the vessel with one tiny dried eucalyptus "
        "sprig, perfect warm reflections."
    ),
}

### 1.2 生成高质量的透明 PNG

In [ ]:
generate_images = True

if generate_images:
    for filename, product_prompt in products.items():
        result = client.images.generate(
            model=IMAGE_MODEL,
            prompt=f"{product_prompt} {brand_prompt}",
            background="transparent",
            size="1024x1536",
            quality="high",
            output_format="png",
        )

        image_bytes = base64.b64decode(result.data[0].b64_json)
        (asset_dir / f"{filename}.png").write_bytes(image_bytes)

### 1.3 检查生成的四款产品

以下是营销网站上使用的原始、未经修改的透明 PNG。

| Wild Fig Eau de Parfum | Meadow Botanical Oil |
| --- | --- |
| ![装在透明琥珀色玻璃瓶中的 Wild Fig Eau de Parfum](images/transparent-image-assets/amber-parfum.png) | ![装在透明琥珀色玻璃瓶中的 Meadow Botanical Oil](images/transparent-image-assets/botanical-oil.png) |
| **Cedar Reed Diffuser** | **Garden After Rain Candle** |
| ![带有细藤条和羽毛状蒲苇纤维的 Cedar Reed Diffuser](images/transparent-image-assets/reed-diffuser.png) | ![装在磨砂琥珀色玻璃容器中的 Garden After Rain Candle](images/transparent-image-assets/garden-candle.png) |

下一个单元格会确认每个文件都包含 alpha 通道，并报告每张图片中完全透明区域所占的比例。

In [ ]:
for filename in products:
    with Image.open(asset_dir / f"{filename}.png") as image:
        if image.mode != "RGBA":
            raise ValueError(f"{filename}.png does not have an alpha channel.")

        transparent_pixels = image.getchannel("A").histogram()[0]
        total_pixels = image.width * image.height
        transparent_percentage = 100 * transparent_pixels / total_pixels

        print(
            f"{filename}.png: {image.width}x{image.height}, "
            f"{transparent_percentage:.2f}% fully transparent"
        )

### 1.4 使用 Codex 构建季节性营销网站

把生成的四张 PNG 交给 Codex，让它为 STILLROOM 构建一个单页店铺网站。完成的网站会把相同的产品放在春、夏、秋、冬四季的营销背景上，允许访问者在单件产品、两件产品或完整系列之间切换，并提供一个棋盘格视图来展示透明效果。

运行下面的单元格，然后把输出连同图片文件一起粘贴到 Codex 中。

In [ ]:
codex_prompt = '''
Build a simple, responsive single-page campaign website for STILLROOM, a
premium botanical home-fragrance brand, using these existing transparent PNGs:

- images/transparent-image-assets/amber-parfum.png
- images/transparent-image-assets/botanical-oil.png
- images/transparent-image-assets/reed-diffuser.png
- images/transparent-image-assets/garden-candle.png

Show the heading "One collection, four seasons." Add Spring, Summer, Autumn,
and Winter controls that change the campaign background and seasonal copy.
Include controls for one product, two products, and the full collection.
Let visitors choose the featured product and rearrange the same transparent
PNGs over every background. Add a "Show transparency" checkerboard toggle.
Keep the page visually clean and centered on the products. Do not generate
replacement images or run a separate background-removal step.
'''.strip()

print(codex_prompt)

### 1.5 查看完成的店铺网站

下面的截图展示了同样的四个透明素材在不同营销背景和产品组合中被复用的效果。

**春季营销活动：绿色背景上的完整四件产品系列。**

![STILLROOM 春季营销网站，在绿色背景上展示全部四张透明产品图片](images/transparent-image-assets/stillroom-spring-campaign.png)

**冬季营销活动：深蓝色背景上的两件产品组合。**

![STILLROOM 冬季营销网站，在深蓝色背景上展示藤条香薰和蜡烛](images/transparent-image-assets/stillroom-winter-campaign.png)

#### 仔细观察难处理的边缘

在不同季节之间切换，仔细查看透明玻璃和磨砂玻璃、香水瓶上的薄纱丝带、香薰的细藤条和发丝般的蒲苇纤维，以及蜡烛的细棉芯。传统的背景去除工具可能会裁掉这些细节、产生光晕，或把半透明材质压平。直接生成 alpha 通道有助于在把同一产品放到截然不同的背景上时保留这些细微的边缘。

## 用例 2：品牌化的企业演示文稿

企业团队通常必须使用带有特定背景颜色或渐变的规定 PowerPoint 主题。带不透明白色背景的 AI 生成图表可能与主题格格不入，而透明 PNG 则能让幻灯片原有的背景保持可见。

### 2.1 定义演示场景和模拟业务数据

虚构的 Northstar Cloud 董事会汇报幻灯片使用两组数据：季度收入，以及合格销售机会的区域分布。这些数值只是示例，并非真实的公司业绩。


In [ ]:
quarterly_revenue = {
    "Q1": 42,
    "Q2": 51,
    "Q3": 63,
    "Q4": 78,
}

regional_pipeline = {
    "North America": 46,
    "Europe": 29,
    "APAC": 25,
}

revenue_growth = (
    (quarterly_revenue["Q4"] - quarterly_revenue["Q1"])
    / quarterly_revenue["Q1"]
)

print(f"Quarterly revenue: {quarterly_revenue} USD millions")
print(f"Regional pipeline: {regional_pipeline} percent")
print(f"Q1-to-Q4 revenue growth: {revenue_growth:.0%}")


### 2.2 定义透明图表提示

要求整个图表都使用透明区域，而不仅仅是图表轮廓之外的部分。对于深色企业主题，还要指定白色或浅色标签、明亮且与品牌协调的颜色，并且不要卡片、边框、面板或填充的绘图区域。

模型会收到准确的模拟数值，但生成的图像内容仍然是栅格图。在使用图表之前，请核实标签、数值和比例是否与源数据一致。对于对数值精度要求很高的正式报告，请以确定性方式渲染图表，而将图像生成用于插图或其他视觉元素。


In [ ]:
revenue_values = ", ".join(
    f"{quarter} = ${amount}M"
    for quarter, amount in quarterly_revenue.items()
)

pipeline_values = ", ".join(
    f"{region} = {percentage}%"
    for region, percentage in regional_pipeline.items()
)

chart_prompts = {
    "quarterly-revenue-growth": (
        "Create one polished enterprise bar chart on a genuinely "
        "transparent background for a dark navy PowerPoint slide. "
        f"Use exactly four ascending bars with these values: {revenue_values}. "
        "Use cobalt blue, bright blue, electric azure, and turquoise bars. "
        "Add white quarter labels below each bar and white dollar-value "
        "labels above each bar. Add a subtle turquoise upward-trend arrow. "
        "Keep the plot area, grid, and space between bars transparent. "
        "Do not add a background, filled panel, frame, title, or card."
    ),
    "regional-pipeline-mix": (
        "Create one polished enterprise doughnut chart on a genuinely "
        "transparent background for a dark navy PowerPoint slide. "
        f"Use exactly these regional pipeline shares: {pipeline_values}. "
        "Use turquoise, cobalt blue, and coral segments, with a white "
        "100% label in the center. Place a white-text regional legend "
        "beside the chart. Keep the doughnut center, legend area, and "
        "all surrounding space transparent. Do not add a background, "
        "filled panel, frame, title, or card."
    ),
}


### 2.3 生成透明 PNG 图表

与产品示例一样，请求 `background="transparent"` 和 `output_format="png"`。`1536x1024` 的横向尺寸为图表标签留出了空间，`quality="high"` 则有助于呈现小号文字。


In [ ]:
generate_charts = True

if generate_charts:
    for filename, prompt in chart_prompts.items():
        result = client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            background="transparent",
            size="1536x1024",
            quality="high",
            output_format="png",
        )

        image_bytes = base64.b64decode(result.data[0].b64_json)
        (asset_dir / f"{filename}.png").write_bytes(image_bytes)


### 2.4 检查生成的图表素材

以下文件是独立的透明 PNG，可以直接放到企业幻灯片主题之上。

| 季度收入 | 区域销售管线构成 |
| --- | --- |
| ![透明柱状图，显示季度收入从 4200 万美元增长到 7800 万美元](images/transparent-image-assets/quarterly-revenue-growth.png) | ![透明环形图，显示北美、欧洲和亚太地区的销售管线占比](images/transparent-image-assets/regional-pipeline-mix.png) |

下一个单元格会检查两个文件是否都包含真正的 alpha 通道，并报告每张图表图片中完全透明区域所占的比例。


In [ ]:
for filename in chart_prompts:
    with Image.open(asset_dir / f"{filename}.png") as image:
        if image.mode != "RGBA":
            raise ValueError(f"{filename}.png does not have an alpha channel.")

        transparent_pixels = image.getchannel("A").histogram()[0]
        total_pixels = image.width * image.height
        transparent_percentage = 100 * transparent_pixels / total_pixels

        if transparent_pixels == 0:
            raise ValueError(f"{filename}.png has no fully transparent pixels.")

        print(
            f"{filename}.png: {image.width}x{image.height}, "
            f"{transparent_percentage:.2f}% fully transparent"
        )


### 2.5 使用 Codex 构建品牌化的 PowerPoint 幻灯片

把两张透明图表 PNG、它们的模拟源数据以及规定的幻灯片主题交给 Codex。要求它保留 alpha 通道，并使用幻灯片自带的背景，而不是添加白色矩形或装饰性背景形状。

运行下面的单元格，然后把输出连同图表图片文件一起粘贴到 Codex 中。


In [ ]:
presentation_prompt = f"""
Create one editable 16:9 PowerPoint slide for a fictional NORTHSTAR CLOUD
FY26 Q4 board review using these existing transparent PNG charts:

- images/transparent-image-assets/quarterly-revenue-growth.png
- images/transparent-image-assets/regional-pipeline-mix.png

Mock quarterly revenue, in USD millions: {quarterly_revenue}
Mock regional pipeline mix, in percent: {regional_pipeline}
Q1-to-Q4 revenue growth: {revenue_growth:.0%}

Use a subtle dark navy-to-teal gradient as the slide's native background.
Add the headline "Growth is accelerating across every region."
Add the subheadline "Revenue climbed 86% over four quarters while pipeline
stayed globally diversified."

Place the quarterly revenue chart on the left and the regional pipeline chart
on the right. Add a heading and short data description above each chart.
Preserve each PNG's real alpha channel so the gradient remains visible behind
both charts. Do not add white image frames, background panels, decorative
shapes, or a separate background-removal step.

Keep the slide clear, polished, and consistent with an enterprise board-deck
template. Add the fictional company name, reporting period, confidentiality
notice, and page number as editable PowerPoint text.
""".strip()

print(presentation_prompt)


### 2.6 查看完成的企业幻灯片

PowerPoint 文件无法直接在 notebook 输出中渲染，因此下图展示的是完成后的单页演示文稿。在底层的 PowerPoint 文件中，两张生成的图表仍然是独立的 PNG 素材。

![企业董事会汇报幻灯片，透明的季度收入图表和区域销售管线图表直接放置在柔和的海军蓝到青绿色渐变背景上](images/transparent-image-assets/enterprise-chart-slide.png)

由于图表包含真正的 alpha 通道，渐变背景在柱子后面、环形图中心以及每个标签周围都清晰可见。这让团队可以把生成的视觉素材添加到规定的企业演示主题中，而不会引入扎眼的白色方框。


## 用例 3：设计模板素材

设计平台需要可复用的应用图标、贴纸和装饰元素，以适配各种版式。以透明背景生成这些元素，用户就可以把同样的素材直接放入任意模板，无需蒙版、抠图或重绘背景。

### 3.1 生成透明设计元素

In [ ]:
design_prompts = {
    "design-app-icon": (
        "Create one premium rounded-square app icon with an abstract "
        "folded paper bird in cobalt blue, lavender, and peach. Keep "
        "everything outside the rounded icon tile transparent. No text, "
        "watermark, shadows, or background."
    ),
    "design-sticker": (
        "Create one illustrated strawberry character sticker with rich "
        "coral-red fruit, sage-green leaves, and a close-fitting cream "
        "die-cut border. Keep everything outside the sticker border "
        "transparent. No text, watermark, shadows, or background."
    ),
    "design-decoration": (
        "Create one delicate editorial botanical sprig with fine "
        "dark-sage branches, muted green leaves, and small coral berries. "
        "Keep all space around and between the thin branches and leaves "
        "transparent. No text, shadows, or background."
    ),
}

generate_design_assets = True

if generate_design_assets:
    for filename, prompt in design_prompts.items():
        result = client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            background="transparent",
            size="1024x1024",
            quality="high",
            output_format="png",
        )

        image_bytes = base64.b64decode(result.data[0].b64_json)
        (asset_dir / f"{filename}.png").write_bytes(image_bytes)

### 3.2 将素材放到一个简单的网站上

每个素材都是独立的透明 PNG，可以随时移动、缩放，或在其他模板上复用。

| 应用图标 | 透明贴纸 | 装饰元素 |
| --- | --- | --- |
| ![透明圆角应用图标，图案为一只抽象的折纸鸟](images/transparent-image-assets/design-app-icon.png) | ![透明草莓角色贴纸，带有贴合轮廓的模切边框](images/transparent-image-assets/design-sticker.png) | ![透明植物装饰，带有纤细的枝条和分离的叶片](images/transparent-image-assets/design-decoration.png) |

示例网页把这三个素材组合在一个彩色模板上，并允许访问者在保持 PNG 位置不变的情况下切换背景。

![简单的设计模板网站，在彩色画布上复用透明应用图标、草莓贴纸和植物装饰](images/transparent-image-assets/design-template-studio.png)

## 用例 4：按需印刷商品

商品平台可以生成一个可直接印刷的透明设计，并将其放到不同的产品上，而无需针对每种服装颜色去除白色矩形。同一件作品可以变成卖家目录中的 T 恤、卫衣或其他商品。

### 4.1 生成可复用的印花设计和空白服装

In [ ]:
merchandise_prompts = {
    "merchandise-design": (
        "Create one premium outdoors-inspired apparel illustration with "
        "an amber setting sun, slate-blue mountains, evergreen pine trees, "
        "a pale turquoise river, and small coral wildflowers. Arrange the "
        "elements into an organic compact circular composition with subtle "
        "screen-print texture. Keep the surrounding area and open spaces "
        "between the graphic elements transparent. No text, bounding "
        "badge, white rectangle, garment, or background."
    ),
    "merchandise-blank-tshirt": (
        "Create one photorealistic premium unisex ivory crew-neck T-shirt "
        "shown completely flat, front-facing, and centered. Leave the "
        "chest completely blank for a print design and keep everything "
        "outside the garment silhouette transparent. No person, hanger, "
        "mannequin, logos, graphics, shadows, or background."
    ),
    "merchandise-blank-sweatshirt": (
        "Create one photorealistic premium forest-green crew-neck "
        "sweatshirt shown completely flat, front-facing, and centered. "
        "Leave the chest completely blank for a print design and keep "
        "everything outside the garment silhouette transparent. No "
        "person, hanger, mannequin, logos, graphics, shadows, or background."
    ),
}

generate_merchandise_assets = True

if generate_merchandise_assets:
    for filename, prompt in merchandise_prompts.items():
        result = client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            background="transparent",
            size="1024x1024",
            quality="high",
            output_format="png",
        )

        image_bytes = base64.b64decode(result.data[0].b64_json)
        (asset_dir / f"{filename}.png").write_bytes(image_bytes)

### 4.2 将同一设计应用到多个产品

印花作品和两件空白服装都是独立的透明图片。将它们分开保存，意味着卖家可以在不同服装类型、产品颜色以及未来的目录版式中复用该设计。

| 可复用的印花设计 | 空白象牙白 T 恤 | 空白森林绿卫衣 |
| --- | --- | --- |
| ![透明户外服饰插画，包含群山、松树和落日](images/transparent-image-assets/merchandise-design.png) | ![空白象牙白圆领 T 恤的透明产品照片](images/transparent-image-assets/merchandise-blank-tshirt.png) | ![空白森林绿圆领卫衣的透明产品照片](images/transparent-image-assets/merchandise-blank-sweatshirt.png) |

使用 Pillow 裁掉空白的透明边距，把同一作品放到每件服装上，并生成一个简单的双产品目录预览。

In [ ]:
from PIL import ImageDraw, ImageFont


def visible_region(image):
    # Remove fully transparent padding without changing visible pixels.
    bounds = image.getchannel("A").getbbox()
    if bounds is None:
        raise ValueError("Expected a transparent PNG with visible artwork.")
    return image.crop(bounds)


canvas = Image.new("RGBA", (1600, 1020), "#f7f5f0")
draw = ImageDraw.Draw(canvas)

heading_font = ImageFont.load_default(size=57)
body_font = ImageFont.load_default(size=21)
label_font = ImageFont.load_default(size=25)
small_font = ImageFont.load_default(size=17)

draw.text((86, 70), "One design, ready to wear.", font=heading_font, fill="#202a27")
draw.text(
    (89, 151),
    "One transparent artwork. Two products. No background removal.",
    font=body_font,
    fill="#68726d",
)

with Image.open(asset_dir / "merchandise-design.png") as image:
    print_design = visible_region(image.convert("RGBA"))

garments = [
    ("merchandise-blank-tshirt", "Everyday T-shirt", "Natural cotton"),
    ("merchandise-blank-sweatshirt", "Essential sweatshirt", "Forest green"),
]

for index, (filename, title, color) in enumerate(garments):
    card_x = 86 + index * 741
    card_y = 221
    card_width = 690
    card_height = 691

    draw.rounded_rectangle(
        (card_x, card_y, card_x + card_width, card_y + card_height),
        radius=28,
        fill="#ffffff",
    )

    with Image.open(asset_dir / f"{filename}.png") as image:
        garment = visible_region(image.convert("RGBA"))

    garment.thumbnail((610, 505), Image.Resampling.LANCZOS)
    garment_x = card_x + (card_width - garment.width) // 2
    garment_y = card_y + 55 + (505 - garment.height) // 2

    artwork = print_design.copy()
    artwork.thumbnail(
        (round(garment.width * 0.32), round(garment.height * 0.32)),
        Image.Resampling.LANCZOS,
    )

    chest_x = (garment.width - artwork.width) // 2
    chest_y = round(garment.height * 0.28)
    garment.alpha_composite(artwork, dest=(chest_x, chest_y))
    canvas.alpha_composite(garment, dest=(garment_x, garment_y))

    draw.text((card_x + 41, card_y + 578), title, font=label_font, fill="#202a27")
    draw.text((card_x + 41, card_y + 620), color, font=small_font, fill="#68726d")

output_path = asset_dir / "merchandise-print-mockups.png"
canvas.convert("RGB").save(output_path, optimize=True)
print(f"Saved merchandise preview to {output_path}")

同一个透明设计自然地融入了两种服装颜色，而空白产品仍可用于其他设计。

![按需印刷目录，展示同一幅透明户外插画分别印在象牙白 T 恤和森林绿卫衣上](images/transparent-image-assets/merchandise-print-mockups.png)

## 评估生成的素材

在具有代表性的营销活动、幻灯片、设计模板和商品样机中，收集人们对透明度、边缘质量、品牌契合度和实用性的反馈。使用具备视觉能力的语言模型，按照清晰的评分标准评估同样的素材，标记出不透明背景、被裁掉的细节、不合适的服装印花位置，以及与源数据不符的图表标签。

## 结论

透明 PNG 素材让团队可以在季节性营销活动、品牌演示文稿、设计模板和商品目录中复用生成的图像，而无需手动去除背景。将这些工作流与人工评估和视觉模型评估结合起来，有助于在使用规模扩大时保持创意产出的一致性。